In [ ]:
import pandas as pd

# Datos del preprocesado
adj_close = pd.read_csv(
    "../Datos_csv/adj_close.csv",
    index_col=0,
    parse_dates=True
)

volume = pd.read_csv(
    "../Datos_csv/volume.csv",
    index_col=0,
    parse_dates=True
)

La metodología se ilustra con un activo representativo. Se trabaja con retornos logarítmicos como variable principal.

In [2]:
import numpy as np
# Retornos
ret_simple = adj_close.pct_change()
ret_log = np.log(adj_close)
ret_log_diff = np.log(adj_close).diff() #diferencia con el dia anterior
dollar_vol = adj_close * volume


In [ ]:
# Lista de ETFs disponibles
tickers = list(adj_close.columns)
print("Número de ETFs:", len(tickers))
print(" Listado:", tickers)

Se analiza la estacionariedad de distintas transformaciones de la serie con el objetivo de identificar la representación más adecuada para la modelización temporal.

In [ ]:
import numpy as np
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tools.sm_exceptions import InterpolationWarning
import warnings
warnings.filterwarnings("ignore", category=InterpolationWarning)

def tests_estacionariedad(serie):
    x = np.asarray(serie, float)
    x = x[~np.isnan(x)]
    # ADF (H0: no estacionaria)
    adf_stat, adf_p, adf_lags, adf_n, adf_crit, adf_icbest = adfuller(x, autolag='AIC')
    
    # KPSS (H0: estacionaria)
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(x, regression='c', nlags='auto')
    if kpss_p <= 0.01 and adf_p>0.01:   # rechazamos KPSS, pero no ADF
        s = "no estacionario"
    elif kpss_p > 0.01 and adf_p<=0.01: # rechazamos ADF, pero no KPSS
        s = "estacionario"
    elif kpss_p > 0.01 and adf_p > 0.01:
        s = "no concluyente"
    else:
        s = "resultados inconsistentes"
    return s

In [5]:
# Estacionariedad (una fila por ETF y por serie)
estacionariedad_rows = []

for etf in tickers:
    # print("\n" + "="*60)
    # print(f"ETF: {etf}")
    # print("="*60)

    #  Extraer series del ETF
  
    serie_adj_close     = adj_close[etf].dropna()
    serie_ret_simple    = ret_simple[etf].dropna()
    serie_log_precio    = ret_log[etf].dropna()
    serie_ret_log_diff  = ret_log_diff[etf].dropna()
    serie_dollar_vol    = dollar_vol[etf].dropna()
    serie_volume        = volume[etf].dropna()

    #  Estacionariedad
    # print(f"\n--- Estacionariedad ({etf}) ---")

    series_a_testear = {
        "Adj_close": serie_adj_close,
        "log(Adj_close)": serie_log_precio,
        "retorno_simple": serie_ret_simple,
        "retorno_logaritmico": serie_ret_log_diff,
        "dollar_volume": serie_dollar_vol,
        "volume": serie_volume
    }

    res_est = []
    for nombre, s in series_a_testear.items():
        if len(s) < 50:
            estado = "insuficientes_datos"
        else:
            estado = tests_estacionariedad(s)

        row = {
            "ETF": etf,
            "Serie": nombre,
            "N_obs": len(s),
            "Inicio": s.index.min().date(),
            "Fin": s.index.max().date(),
            "Conclusión": estado
        }
        estacionariedad_rows.append(row)
        res_est.append(row)

    # display(pd.DataFrame(res_est).head(2))

ETFs eliminados por no ser estacionarios en retornos.

In [6]:

# Convertimos la lista acumulada en DataFrame
estacionariedad_df = pd.DataFrame(estacionariedad_rows)

# ETFs válidos
etfs_validos = []

# ETFs eliminados
etfs_eliminados = []

for etf in tickers:
    
    # Filtramos solo las filas de ese ETF
    df_etf = estacionariedad_df[estacionariedad_df["ETF"] == etf]
    
    # Sacamos las conclusiones de los retornos
    ret_simple_estado = df_etf[df_etf["Serie"] == "retorno_simple"]["Conclusión"].values
    ret_log_estado    = df_etf[df_etf["Serie"] == "retorno_logaritmico"]["Conclusión"].values
    
    # Comprobamos que existan y sean estacionarios
    if (
        len(ret_simple_estado) > 0 and
        len(ret_log_estado) > 0 and
        ret_simple_estado[0] == "estacionario" or
        ret_log_estado[0] == "estacionario"
    ):
        etfs_validos.append(etf)
    else:
        etfs_eliminados.append(etf)

# Mostrar resultados
print("Número total ETFs originales:", len(tickers))
print("Número ETFs válidos:", len(etfs_validos))
print("Número ETFs eliminados:", len(etfs_eliminados))

print("\nETFs eliminados:")
display(pd.DataFrame(etfs_eliminados, columns=["ETF eliminado"]))

# Sobrescribimos la lista original si quieres trabajar solo con los válidos
tickers_filtrados = etfs_validos


Número total ETFs originales: 63
Número ETFs válidos: 58
Número ETFs eliminados: 5

ETFs eliminados:


,ETF eliminado
0,BIL
1,EWY
2,GLD
3,IAU
4,SHY


Los retornos son estacionarios; precio y log(precio) no. Se trabaja con retornos logarítmicos porque son directamente interpretables y compatibles con la optimización de carteras posterior.

In [7]:
import json
import os

resultado = {
    "tickers_filtrados": tickers_filtrados,
    "etfs_eliminados": etfs_eliminados
}

out_path = "../Datos_csv/tickers_estacionalidad.json"
with open(out_path, 'w', encoding='utf-8') as fout:
    json.dump(resultado, fout, indent=2, ensure_ascii=False)

print(f"Guardado en: {out_path}")
print(f"ETFs validos  : {len(tickers_filtrados)}")
print(f"ETFs eliminados: {etfs_eliminados}")

Guardado en: ../Datos_csv/tickers_estacionalidad.json
ETFs validos  : 58
ETFs eliminados: ['BIL', 'EWY', 'GLD', 'IAU', 'SHY']
